# Nedostajući podaci - zamjena

## 1.) Linearna regresija

In [9]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# =========================================================
# Originalni potpuni dataset
# =========================================================
df_true = pd.DataFrame({
    "n": [1, 2, 3, 4, 5],
    "x": [0.55, 0.75, 0.32, 0.21, 0.43],
    "y": [0.53, 0.37, 0.83, 0.86, 0.54]
})

# ukljanjanje vrijednosti za primjer:
df_missing = df_true.copy()
df_missing.loc[1, "x"] = np.nan
df_missing.loc[3, "y"] = np.nan

print("ORIGIGI")
print(df_true.round(2).to_string(index=False))

print("\nPodaci s nedostajućim vrijednostima")
print(df_missing.round(2).to_string(index=False))


# =========================================================
# 1) Ručna zamjena LINEARNOM REGRESIJOM
#    A: impute missing x from y   => x = a*y + b
#    B: impute missing y from x   => y = a*x + b
# =========================================================

def manual_regression_impute_target_from_feature(df, feature_col, target_col):
    """
    Fits target = a*feature + b on complete rows,
    then imputes missing target where feature is available.
    """
    complete = df[df[feature_col].notna() & df[target_col].notna()].copy()
    missing_target = df[df[target_col].isna() & df[feature_col].notna()].copy()

    n = len(complete)
    sx = complete[feature_col].sum()
    sy = complete[target_col].sum()
    sx2 = (complete[feature_col] ** 2).sum()
    sxy = (complete[feature_col] * complete[target_col]).sum()

    a = (n * sxy - sx * sy) / (n * sx2 - sx ** 2)
    b = (sy / n) - a * (sx / n)

    preds = a * missing_target[feature_col] + b

    result = df.copy()
    result.loc[missing_target.index, target_col] = preds

    return result, a, b, missing_target.index, preds


# A) Impute x from y
df_manual_x, a_x, b_x, idx_x, preds_x = manual_regression_impute_target_from_feature(
    df_missing, feature_col="y", target_col="x"
)

# B) Then impute y from x
df_manual_xy, a_y, b_y, idx_y, preds_y = manual_regression_impute_target_from_feature(
    df_manual_x, feature_col="x", target_col="y"
)

print("\n1) Ručna zamjena LINEARNOM REGRESIJOM")
print(f"Za nedostajući x: x = {a_x:.6f} * y + {b_x:.6f}")
for idx, pred in zip(idx_x, preds_x):
    print(f"Redak n={df_missing.loc[idx, 'n']}: izračunati x = {pred:.4f}")

print(f"Za nedostajući y: y = {a_y:.6f} * x + {b_y:.6f}")
for idx, pred in zip(idx_y, preds_y):
    print(f"Redak n={df_missing.loc[idx, 'n']}: izračunati y = {pred:.4f}")

print("\nNakon provedene zamjene:")
print(df_manual_xy.round(4).to_string(index=False))


# =========================================================
# 2) LinearRegression()
# =========================================================

df_lr = df_missing.copy()

# A) Impute x from y
train_x = df_lr[df_lr["x"].notna() & df_lr["y"].notna()]
pred_x_rows = df_lr[df_lr["x"].isna() & df_lr["y"].notna()]

lr_x = LinearRegression()
lr_x.fit(train_x[["y"]], train_x["x"])
x_preds = lr_x.predict(pred_x_rows[["y"]])

df_lr.loc[pred_x_rows.index, "x"] = x_preds

# B) Impute y from x
train_y = df_lr[df_lr["x"].notna() & df_lr["y"].notna()]
pred_y_rows = df_lr[df_lr["y"].isna() & df_lr["x"].notna()]

lr_y = LinearRegression()
lr_y.fit(train_y[["x"]], train_y["y"])
y_preds = lr_y.predict(pred_y_rows[["x"]])

df_lr.loc[pred_y_rows.index, "y"] = y_preds

print("\n2) LinearRegression()")
print(f"Za nedostajući: x = {lr_x.coef_[0]:.6f} * y + {lr_x.intercept_:.6f}")
for idx, pred in zip(pred_x_rows.index, x_preds):
    print(f"Redak n={df_missing.loc[idx, 'n']}: izračunati x = {pred:.4f}")

print(f"Za nedostajući y: y = {lr_y.coef_[0]:.6f} * x + {lr_y.intercept_:.6f}")
for idx, pred in zip(pred_y_rows.index, y_preds):
    print(f"Redak n={df_missing.loc[idx, 'n']}: izračunati y = {pred:.4f}")

print("\nDataset nakon LinearRegression():")
print(df_lr.round(4).to_string(index=False))


# =========================================================
# 3) IterativeImputer
# =========================================================

imp = IterativeImputer(
    estimator=LinearRegression(),
    max_iter=20,
    random_state=42
)

df_iter = df_missing.copy()
df_iter[["x", "y"]] = imp.fit_transform(df_iter[["x", "y"]])

print("\n3) IterativeImputer")
print(df_iter.round(4).to_string(index=False))


# =========================================================
# USPOREDBA
# =========================================================

comparison = pd.DataFrame({
    "n": df_true["n"],
    "true_x": df_true["x"],
    "true_y": df_true["y"],
    "manual_x": df_manual_xy["x"],
    "manual_y": df_manual_xy["y"],
    "lr_x": df_lr["x"],
    "lr_y": df_lr["y"],
    "iter_x": df_iter["x"],
    "iter_y": df_iter["y"],
})

print("\nUSPOREDBA")
print(comparison.round(4).to_string(index=False))

ORIGIGI
 n    x    y
 1 0.55 0.53
 2 0.75 0.37
 3 0.32 0.83
 4 0.21 0.86
 5 0.43 0.54

Podaci s nedostajućim vrijednostima
 n    x    y
 1 0.55 0.53
 2  NaN 0.37
 3 0.32 0.83
 4 0.21  NaN
 5 0.43 0.54

1) Ručna zamjena LINEARNOM REGRESIJOM
Za nedostajući x: x = -0.586108 * y + 0.804535
Redak n=2: izračunati x = 0.5877
Za nedostajući y: y = -1.455265 * x + 1.254267
Redak n=4: izračunati y = 0.9487

Nakon provedene zamjene:
 n      x      y
 1 0.5500 0.5300
 2 0.5877 0.3700
 3 0.3200 0.8300
 4 0.2100 0.9487
 5 0.4300 0.5400

2) LinearRegression()
Za nedostajući: x = -0.586108 * y + 0.804535
Redak n=2: izračunati x = 0.5877
Za nedostajući y: y = -1.455265 * x + 1.254267
Redak n=4: izračunati y = 0.9487

Dataset nakon LinearRegression():
 n      x      y
 1 0.5500 0.5300
 2 0.5877 0.3700
 3 0.3200 0.8300
 4 0.2100 0.9487
 5 0.4300 0.5400

3) IterativeImputer
 n      x      y
 1 0.5500 0.5300
 2 0.6031 0.3700
 3 0.3200 0.8300
 4 0.2100 0.9409
 5 0.4300 0.5400

USPOREDBA
 n  true_x  true_y  

## 2.) K-najbližih susjeda

In [10]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer

df_true = pd.DataFrame({
    "n": [1, 2, 3, 4, 5],
    "x": [0.55, 0.75, 0.32, 0.21, 0.43],
    "y": [0.53, 0.37, 0.83, 0.86, 0.54]
})

df_missing = df_true.copy()
df_missing.loc[1, "x"] = np.nan   # row n=2
df_missing.loc[3, "y"] = np.nan   # row n=4

print("ORIGIGI")
print(df_true.round(2).to_string(index=False))

print("\nPodaci s nedostajućim vrijednostima")
print(df_missing.round(2).to_string(index=False))

# =========================================================
# KNN imputation
# =========================================================

imputer = KNNImputer(n_neighbors=2)

imputed_array = imputer.fit_transform(df_missing[["x", "y"]])

df_knn = df_missing.copy()
df_knn[["x", "y"]] = imputed_array

print("\nDATA nakon KNN imputation")
print(df_knn.round(4).to_string(index=False))

# =========================================================
# USPOREDBA
# =========================================================
comparison = pd.DataFrame({
    "n": df_true["n"],
    "true_x": df_true["x"],
    "true_y": df_true["y"],
    "missing_x": df_missing["x"],
    "missing_y": df_missing["y"],
    "knn_x": df_knn["x"],
    "knn_y": df_knn["y"]
})

print("\nUSPOREDBA")
print(comparison.round(4).to_string(index=False))

ORIGIGI
 n    x    y
 1 0.55 0.53
 2 0.75 0.37
 3 0.32 0.83
 4 0.21 0.86
 5 0.43 0.54

Podaci s nedostajućim vrijednostima
 n    x    y
 1 0.55 0.53
 2  NaN 0.37
 3 0.32 0.83
 4 0.21  NaN
 5 0.43 0.54

DATA nakon KNN IMPUTATION
 n    x     y
 1 0.55 0.530
 2 0.49 0.370
 3 0.32 0.830
 4 0.21 0.685
 5 0.43 0.540

USPOREDBA
 n  true_x  true_y  missing_x  missing_y  knn_x  knn_y
 1    0.55    0.53       0.55       0.53   0.55  0.530
 2    0.75    0.37        NaN       0.37   0.49  0.370
 3    0.32    0.83       0.32       0.83   0.32  0.830
 4    0.21    0.86       0.21        NaN   0.21  0.685
 5    0.43    0.54       0.43       0.54   0.43  0.540
